In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
DATASET_DIRECTORY = "../data/airbus-ship-detection"

FILEPATH = f"{DATASET_DIRECTORY}/train_ship_segmentations_v2.csv"

df = pd.read_csv(FILEPATH)
print("Количество записей:", len(df))
print("Количество изображений:", df["ImageId"].nunique())
print("Количество размеченных кораблей:", df["EncodedPixels"].notna().sum())
df = df.dropna(subset=["EncodedPixels"])
print("Количество записей после удаления записей без кораблей:", len(df))

In [ ]:
def decode_rle(rle, height, width):
    mask = np.zeros(height*width, dtype=np.uint8)

    values = np.fromstring(rle, sep=" ",dtype=np.int64)
    starts = values[::2] - 1
    lengths = values[1::2]

    for start, length in zip(starts, lengths):
        mask[start:start+length] = 1

    return mask.reshape((height, width), order='F')

In [ ]:
counts = df.groupby("ImageId").size()
image_id = counts[counts >=3].index[0]

rles = df.loc[
    df["ImageId"] == image_id,
    "EncodedPixels"
]

h, w = 768, 768

instance_map = np.zeros((h, w), dtype=np.int32)

for instance_id, rle in enumerate(rles, start=1): # 0 - background, 1 - first ship, 2 - second ship....
    mask = decode_rle(rle, h, w)
    instance_map[mask == 1] = instance_id

image = plt.imread(f"{DATASET_DIRECTORY}/train_v2/{image_id}")

fig, axes = plt.subplots(1, 2, figsize = (10, 6))

# Скрываем фон (пиксели с меткой 0)
masked_instances = np.ma.masked_equal(instance_map, 0)

axes[0].imshow(image)
axes[1].imshow(image)
axes[1].imshow(masked_instances, cmap="turbo", alpha=0.5)

axes[0].set_title(f"{image_id}: Original")
axes[1].set_title(f"{image_id}: {len(rles)} ships")
for ax in axes:
    ax.axis("off")

plt.show()